# 05 — Agent 3 : verbalisation des portions V2

L'Agent 3 reçoit directement la sortie structurée du notebook 04.

Exemples d'états possibles :

```text
montee + forte_pente + lacets + foret
descente + pente_douce + foret
montee + lacets
foret
```

Les contradictions `montee+descente` et `douce+forte` ont déjà été
éliminées par l'Agent 2.

L'Agent 3 ne doit rien inventer.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import requests

## 1. Charger les portions du notebook 04

In [ ]:
portions_path = Path(
    "data/output/agent2_portions.json"
)

if not portions_path.exists():
    raise FileNotFoundError(
        "Exécute d'abord le notebook 04."
    )

with portions_path.open(
    "r",
    encoding="utf-8",
) as f:
    portions = json.load(f)

print("Portions chargées :", len(portions))

## 2. Client Ollama

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "llama3.1:8b"


def ask_ollama(prompt):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            "stream": False,
            "options": {
                "temperature": 0
            },
        },
        timeout=120,
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Ollama HTTP "
            f"{response.status_code}: "
            f"{response.text}"
        )

    return (
        response
        .json()["message"]["content"]
        .strip()
    )

## 3. Verbaliser une portion

Règles importantes :

- `montee + forte_pente` → formulation naturelle de type **forte montée** ;
- `descente + pente_douce` → **descente douce** ;
- `foret` signifie seulement passage en zone boisée OSM ;
- ne jamais déduire automatiquement `ombre`, `vue`, `sentier`, etc.

In [ ]:
def humanize_portion(portion):
        payload = {
            "start_m": portion["start_m"],
            "end_m": portion["end_m"],
            "slope_direction": portion.get(
                "slope_direction"
            ),
            "slope_intensity": portion.get(
                "slope_intensity"
            ),
            "geometry_features": portion.get(
                "geometry_features",
                [],
            ),
            "terrain_features": portion.get(
                "terrain_features",
                [],
            ),
        }

        prompt = f'''
Tu décris une portion d'itinéraire en français.

Données structurées :
{json.dumps(payload, ensure_ascii=False, indent=2)}

Règles obligatoires :
- écris UNE phrase concise ;
- n'invente aucune information absente ;
- ne cite pas les noms techniques des champs ;
- si direction=montee et intensité=forte_pente,
  formule naturellement "forte montée" ;
- si direction=descente et intensité=pente_douce,
  formule naturellement "descente douce" ;
- si terrain contient "foret", tu peux dire que la portion
  traverse ou progresse en forêt / zone boisée ;
- "foret" ne permet PAS d'affirmer qu'il y a de l'ombre ;
- "lacets" peut être formulé comme une succession de lacets ;
- conserve fidèlement les distances ;
- aucune information géographique ou paysagère supplémentaire.

Réponds uniquement avec la phrase finale.
'''

        return ask_ollama(prompt)

## 4. Générer toutes les descriptions

In [ ]:
descriptions = []

for portion in portions:
    text = humanize_portion(portion)

    descriptions.append({
        "start_m": portion["start_m"],
        "end_m": portion["end_m"],
        "annotations": portion.get(
            "annotations",
            [],
        ),
        "text": text,
    })

    print(
        f'{portion["start_m"]:.0f}–'
        f'{portion["end_m"]:.0f} m : '
        f'{text}'
    )

## 5. Export des descriptions

In [ ]:
output_dir = Path("data/output")
output_dir.mkdir(parents=True, exist_ok=True)

descriptions_json = (
    output_dir / "agent3_descriptions.json"
)

descriptions_csv = (
    output_dir / "agent3_descriptions.csv"
)

with descriptions_json.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        descriptions,
        f,
        ensure_ascii=False,
        indent=2,
    )

pd.DataFrame(
    descriptions
).to_csv(
    descriptions_csv,
    index=False,
)

print("Exports Agent 3 :")
print("-", descriptions_json)
print("-", descriptions_csv)